# 01 Raw BIDS import

This notebook only imports source recordings into the configured raw BIDS root.

It converts regular MEG recordings from `sourcedata/` and empty-room recordings
from `sourcedata/emptyroom/` into the BIDS dataset configured by
`paths.bids_root`, for example `rawdata/`.

Manual bad-channel QC and trigger-event writing are handled in the next notebook.
This avoids a common first-run problem: before conversion, raw BIDS recordings do
not yet exist, so recording selection for QC would be empty until the notebook is
run a second time.


## Project root and raw BIDS root

This notebook treats the outer project folder as the analysis workspace. The raw BIDS dataset is `config.paths.bids_root`, normally `rawdata/`.

Expected layout:

```text
{{ project_name }}/
  README.md                  # project README
  configs/local.yaml
  notebooks/
  sourcedata/                 # original acquisition exports
  rawdata/                    # raw BIDS dataset root
    README
    dataset_description.json
    participants.tsv
    participants.json
    sub-*
    sub-emptyroom/
  derivatives/
```

Do not place `dataset_description.json`, `participants.tsv`, `participants.json`, or BIDS `sub-*` folders in the outer project root. They belong under `rawdata/`.


## Setup


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from meeg_pipeline.config import load_config
from meeg_pipeline.conversion import (
    convert_empty_room_source_recordings_to_bids,
    convert_source_recordings_to_bids,
    conversion_results_to_dataframe,
)
from meeg_pipeline.sourcedata import (
    discover_empty_room_source_recordings,
    discover_empty_room_source_recordings_with_issues,
    discover_source_recordings,
    discover_source_recordings_with_issues,
    empty_room_sourcedata_overview_to_dataframe,
    make_target_bids_path,
)
from meeg_pipeline.workflow import (
    existing_output_policy_for_step,
    iter_recordings,
    recordings_to_dataframe,
    should_overwrite,
)

def find_project_root(start: Path | None = None) -> Path:
    """Find project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)
print("BIDS_ROOT:", config.paths.bids_root)


In [ ]:
# Check BIDS-root metadata placement.
required_bids_root_files = ["dataset_description.json", "participants.tsv"]
optional_bids_root_files = ["participants.json", "README"]

print("Project root:", PROJECT_ROOT)
print("Raw BIDS root:", config.paths.bids_root)

for filename in required_bids_root_files:
    path = config.paths.bids_root / filename
    print(f"{filename}:", "found" if path.exists() else "MISSING", "-", path)

for filename in optional_bids_root_files:
    path = config.paths.bids_root / filename
    print(f"{filename}:", "found" if path.exists() else "not present", "-", path)


## Overwrite policy

Existing raw BIDS files are skipped by default. Use `OVERWRITE_STEPS` only when
you intentionally want to recreate raw BIDS files from `sourcedata/`.


In [ ]:
OVERWRITE_STEPS = []

pd.DataFrame(
    [
        {
            "step": "convert_to_bids",
            "overwrite": should_overwrite("convert_to_bids", OVERWRITE_STEPS),
            "policy": existing_output_policy_for_step(
                "convert_to_bids",
                OVERWRITE_STEPS,
            ),
        },
        {
            "step": "empty_room_to_bids",
            "overwrite": should_overwrite("empty_room_to_bids", OVERWRITE_STEPS),
            "policy": existing_output_policy_for_step(
                "empty_room_to_bids",
                OVERWRITE_STEPS,
            ),
        },
    ]
)


## Discover source recordings

This uses the configured `paths.sourcedata_root`.

If `sourcedata_root` is missing or if some source folders are incomplete, this section reports status instead of interrupting the notebook.


In [ ]:
source_recordings = discover_source_recordings(config)

source_rows = []
for recording in source_recordings:
    target_bids_path = make_target_bids_path(config, recording)
    source_rows.append(
        {
            "subject": recording.subject,
            "session": recording.session,
            "task": recording.task,
            "run": recording.run,
            "source_path": str(recording.source_path),
            "target_path": str(target_bids_path.fpath),
            "target_exists": target_bids_path.fpath.exists(),
        }
    )

pd.DataFrame(source_rows)


### Optional: source discovery issues

Use this if you want to inspect folders that were skipped during sourcedata discovery.


In [ ]:
try:
    _recordings, source_issues = discover_source_recordings_with_issues(config)
except AttributeError:
    source_issues = []

pd.DataFrame(
    [
        {
            "path": str(issue.path),
            "status": issue.status,
            "message": issue.message,
        }
        for issue in source_issues
    ]
)


## Convert sourcedata to raw BIDS

Existing targets are skipped by default.


In [ ]:
conversion_policy = existing_output_policy_for_step(
    "convert_to_bids",
    OVERWRITE_STEPS,
)

conversion_results = convert_source_recordings_to_bids(
    config,
    source_recordings,
    on_existing=conversion_policy,
)

pd.DataFrame(
    [
        {
            "status": result.status,
            "message": result.message,
            "source_path": result.source_path,
            "target_path": result.target_path,
        }
        for result in conversion_results
    ]
)


## Discover empty-room source recordings

Empty-room files are discovered from `empty_room.sourcedata_root`, for example:

```text
sourcedata/emptyroom/ses-20250313/<arbitrary-file-name>.fif
```

The source filename may be arbitrary. The session is inferred from the `ses-*` folder. Empty-room recordings are converted to `sub-emptyroom/ses-*/meg/*_task-noise_meg.fif` and do **not** need events.tsv files.

In [ ]:
empty_room_source_recordings = discover_empty_room_source_recordings(config)
empty_room_sourcedata_overview_to_dataframe(config)

### Optional: empty-room discovery issues

In [ ]:
try:
    _empty_room_recordings, empty_room_issues = discover_empty_room_source_recordings_with_issues(config)
except AttributeError:
    empty_room_issues = []

pd.DataFrame(
    [
        {
            "path": str(issue.path),
            "status": issue.status,
            "message": issue.message,
        }
        for issue in empty_room_issues
    ]
)

## Convert empty-room sourcedata to raw BIDS

Existing empty-room BIDS targets are skipped by default. Empty-room conversion is controlled by `OVERWRITE_STEPS = ["empty_room_to_bids"]`.

In [ ]:
empty_room_conversion_policy = existing_output_policy_for_step(
    "empty_room_to_bids",
    OVERWRITE_STEPS,
)

empty_room_conversion_results = convert_empty_room_source_recordings_to_bids(
    config,
    empty_room_source_recordings,
    on_existing=empty_room_conversion_policy,
)

conversion_results_to_dataframe(empty_room_conversion_results)

## Raw BIDS recordings after import

After conversion, this cell re-reads the configured raw BIDS dataset. If this
table contains the expected recordings, continue with
`02_bad_channels_and_events.ipynb`.


In [ ]:
# Re-read the raw BIDS dataset after conversion.
# This is intentionally done at the end of the import notebook, so the next
# notebook can select recordings from the freshly created BIDS files.
raw_bids_recordings = list(iter_recordings(config, subjects="all"))
recordings_to_dataframe(raw_bids_recordings)
